# DP-SGD architecture ablation — how much privacy comes from BN→GN?

Trains EEGNet with the architectural surgery Opacus's `ModuleValidator.fix` performs (BatchNorm → GroupNorm) and the same SGD optimizer DP-SGD uses, but **without** the per-sample gradient clipping or Gaussian noise — i.e. ε=∞ (no DP guarantee). Then runs A1 closed-set re-ID against its embeddings.

Pair the result with `results/02_closed_set_reid.json` (AdamW+BN baseline) and `results/10_d3_dp_sgd.json` (DP at ε=3) to break down where D3's privacy comes from: architecture-vs-noise.

Send back `results/19_dp_sgd_arch_ablation.json`.

**Runtime → L4 GPU.** Expected wall ~25-35 min.

**Don't `Save a copy in GitHub` from Colab.**

## 1. Clone repo

In [ ]:
!rm -rf /content/bci-identity-leakage
!git clone https://github.com/manrajmondair/bci-identity-leakage.git /content/bci-identity-leakage
%cd /content/bci-identity-leakage
!git rev-parse HEAD

## 2. Confirm GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
print('torch:', torch.__version__, '| cuda:', torch.cuda.is_available(),
      '| device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')

## 3. Install deps

In [ ]:
import torch
tv = torch.__version__.split('+')[0]
!pip install --quiet "torchaudio=={tv}"
!pip install --quiet mne moabb pyriemann braindecode opacus httpx tqdm rich

## 4. Prefetch PhysioNet imagery (~2 min)

In [ ]:
!python -m data.prefetch_direct --runs imagery --workers 8

## 5. Run experiment

Expected wall: ~25-35 min.

In [ ]:
!PYTHONUNBUFFERED=1 python -u -m experiments.19_dp_sgd_arch_ablation --all

## 6. Emit run metadata + result JSON

In [ ]:
import json, os, subprocess, datetime, platform, sys
import torch
PROJECT_DIR = '/content/bci-identity-leakage'
if os.path.isdir(PROJECT_DIR): os.chdir(PROJECT_DIR)
EXPERIMENT = "experiments.19_dp_sgd_arch_ablation"
ARGS = ['--all']
RESULTS = 'results/19_dp_sgd_arch_ablation.json'
EXTRA_OUTPUTS = []
TAG = "dp_sgd_arch"
try:
    git_sha = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=PROJECT_DIR).decode().strip()
except Exception as exc:
    print(f'  git unavailable ({exc}); using "unknown"'); git_sha = 'unknown'
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'
now_utc = datetime.datetime.utcnow().replace(microsecond=0).isoformat() + 'Z'
run_id = now_utc.replace(':','').replace('-','').rstrip('Z') + f'_{TAG}_{git_sha[:7]}'
meta = {'run_id': run_id, 'experiment': EXPERIMENT, 'args': ARGS,
        'git_sha': git_sha, 'hardware': f'Colab {gpu}',
        'platform': platform.platform(), 'torch_version': torch.__version__,
        'completed_at_utc': now_utc, 'outputs': [RESULTS] + EXTRA_OUTPUTS}
os.makedirs(f'runs/{run_id}', exist_ok=True)
with open(f'runs/{run_id}/meta.json', 'w') as f: json.dump(meta, f, indent=2)
print('=== run metadata ==='); print(json.dumps(meta, indent=2)); print()
if not os.path.exists(RESULTS):
    sys.exit(f'!!! {RESULTS} missing — run cell did not finish. Re-run the run cell, then this cell.')
print(f'=== {RESULTS} ==='); print(open(RESULTS).read())


## 7. Download artifacts

In [ ]:
from google.colab import files
files.download('results/19_dp_sgd_arch_ablation.json')

files.download(f'runs/{run_id}/meta.json')
